## Model accuracy
Import libraries

In [ ]:
import os
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from silx.io.dictdump import h5todict

Define inputs

In [ ]:
model_name = "meno" # "meno_nocorr"

In [ ]:
# Models
models = {
  "meno": "./../03.merged/run01/saving/pollu/SavedModel/",
  "meno_nocorr": "./../02.y_lin/run01.nocorr/saving/pollu_y_lin/SavedModel/"
}
# Paths
path_to_datagen = "./../00.datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": models[model_name],
  "indices": path_to_datagen + "/data/physics/species_indices.hdf5",
  "saving": f"./output/error/{model_name}"
}
os.makedirs(paths["saving"], exist_ok=True)
# Error
# > Epsilon value
eps = 1e-10
# > Confidence interval
alpha = 0.95
uncert_name = f"ci_{int(100*alpha)}" if (alpha > 0) else "std"

In [ ]:
# Time grid
tlim = [1e-7, 60.0]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Define data indices

In [ ]:
# Data indices
species_indices = h5todict(paths["indices"])
if (model_name == "meno"):
  indices = {
    "x": [
      species_indices["lin"],       # Linear species at time t=0
      species_indices["nlin"],      # Nonlinear species at time t=0
      species_indices["inp"],       # Input species at time t=0
      np.array([20])                # Time
    ],
    "y": [
      species_indices["lin"]+21,    # Linear species at time t
      species_indices["nlin"]+21    # Nonlinear species at time t
    ]
  }
elif (model_name == "meno_nocorr"):
  indices = {
    "x": [
      species_indices["lin"],       # Linear species at time t=0
      species_indices["nlin"],      # Nonlinear species at time t=0
      species_indices["inp"],       # Input species at time t=0
      np.array([20]),               # Time
      species_indices["nlin"]+21    # Nonlinear species at time t
    ],
    "y": [
      species_indices["lin"]+21     # Linear species at time t
    ]
  }
else:
  raise ValueError()

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Set variables names

np.concatenate(indices["y"])

In [ ]:
names = np.concatenate(indices["y"])-20
names = [f"y{i}" for i in names]
names

Compute error

In [ ]:
error = {"l2": [], "mape": []}
for idata in data.values():
  _, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices, t, t_index=3)
  error["l2"].append(utils.error_l2(y_true, y_pred, eps=eps))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=eps))

Plot and save error

In [ ]:
xticks = [1] + list(range(5,21,5))
xticklabels = [str(i) for i in xticks]

In [ ]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  mean, uncert = utils.compute_stats(kerr, axis=0, alpha=alpha)
  kerr = np.vstack([mean, uncert])
  # Plotting error
  utils.plot_err_discrete(
    x=np.concatenate(indices["y"])-20,
    mean=mean,
    uncert=uncert,
    alpha=alpha,
    xlim=[0.5,20.5],
    xticks=xticks,
    xticklabels=xticklabels,
    labels=[r"Species", r"Error [\%]"],
    scales=["linear", "linear"],
    legend_loc="best",
    figname=paths["saving"]+f"/err_dist_{k}.pdf",
    save=True,
    show=False
  )
  # Saving
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", uncert_name])
  kerr = kerr.sort_values(by=['mean'], axis=1)
  kerr.to_csv(paths["saving"]+f"/{model_name}_{k}.csv", float_format="%.3f")
  utils.write_latex(
    error=kerr,
    filename=paths["saving"]+f"/{model_name}_{k}_latex.txt",
    names_stats=["mean", uncert_name]
  )